## 0. Imports and configuration

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, UnidentifiedImageError

try:
    RESAMPLING_METHOD = Image.Resampling.LANCZOS
except AttributeError:
    RESAMPLING_METHOD = Image.LANCZOS
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Ambiente: {"Google Colab" if IN_COLAB else "Local"}')

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR  = Path('/content/drive/MyDrive/Doutorado-V3/Trilha3-V2')
    IMGS_DIR  = Path('/content/drive/MyDrive/Doutorado-V3/Data/Imgs')
    CSV_PATH  = Path('/content/drive/MyDrive/Doutorado-V3/Data/Imgs-anotadas/Planilha sem título - IMAGENS ROTULADAS.csv')
else:
    BASE_DIR  = Path('e:/Doutorado-V3/Trilha3-V2')
    IMGS_DIR  = Path('e:/Doutorado-V3/Data/Imgs')
    CSV_PATH  = Path('e:/Doutorado-V3/Data/Imgs-anotadas/Planilha sem título - IMAGENS ROTULADAS.csv')

SPLITS_DIR  = BASE_DIR / 'splits'
RESULTS_DIR = BASE_DIR / 'results' / 'audit'
FIGS_DIR    = BASE_DIR / 'figs' / 'eda'

for d in [RESULTS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORE_LABELS     = ['ENANTEMA', 'PÓLIPO', 'ÚLCERA', 'EROSÃO', 'MICRONODULARIDADE']
ARTIFACT_LABELS = ['SALIVA', 'LUZ']
ALL_LABELS      = CORE_LABELS + ARTIFACT_LABELS

print('Caminhos OK.')
print(f'IMGS_DIR existe: {IMGS_DIR.exists()}')
print(f'CSV existe     : {CSV_PATH.exists()}')

## 1. Reading and cleaning original CSV

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(f'Shape bruto : {df_raw.shape}')
print(f'Colunas     : {list(df_raw.columns)}')
print(f'\nPrimeiras linhas:')
df_raw.head(3)

In [ ]:
df_raw = df_raw.rename(columns={'Coluna 1': 'image_name'})

label_cols = ['NORMAL','ALTERADO','SALIVA','LUZ','ENANTEMA','PÓLIPO',
              'ÚLCERA','EROSÃO','MICRONODULARIDADE','ECTASIA VASCULAR','NEOPLASIA']

df = df_raw.copy()
for col in label_cols:
    df[col] = df[col].map({1: 1, 2: 0})

_on_disk = {p.name.lower() for p in IMGS_DIR.iterdir() if p.is_file()}
df['has_file'] = df['image_name'].apply(lambda x: str(x).lower() in _on_disk)

df_valid = df[df['has_file']].copy().reset_index(drop=True)

print(f'Total no CSV            : {len(df)}')
print(f'Com arquivo em disco    : {len(df_valid)}')
print(f'Sem arquivo (órfãos CSV): {len(df) - len(df_valid)}')
print()
print('Valores missing por coluna:')
print(df_valid[label_cols].isna().sum().to_string())

In [ ]:
print(f'\n{"Rótulo":<22} {"Presentes":>10} {"Ausentes":>10} {"Missing":>10} {"Prevalência":>12} {"IR":>8}')
print('-' * 74)

label_stats = {}
for col in label_cols:
    n_present = int(df_valid[col].sum())
    n_missing = int(df_valid[col].isna().sum())
    n_valid   = len(df_valid) - n_missing
    n_absent  = n_valid - n_present
    prev      = n_present / n_valid if n_valid > 0 else 0
    ir        = n_absent / n_present if n_present > 0 else float('inf')
    label_stats[col] = {
        'present': n_present, 'absent': n_absent,
        'missing': n_missing, 'prevalence': round(prev, 4), 'ir': round(ir, 2)
    }
    print(f'{col:<22} {n_present:>10} {n_absent:>10} {n_missing:>10} '
          f'{prev:>11.1%} {ir:>8.1f}')

## 2. Consistency with cleaned splits

In [ ]:
all_dfs = []
for f in range(5):
    for s in ['train', 'val', 'test']:
        all_dfs.append(pd.read_csv(SPLITS_DIR / f'fold_{f}_{s}.csv'))
df_split = pd.concat(all_dfs).drop_duplicates(subset=['image_name'])

in_splits   = set(df_split['image_name'])
in_csv      = set(df_valid['image_name'])
only_splits = in_splits - in_csv
only_csv    = in_csv - in_splits

df_merge = df_split.merge(
    df_valid[['image_name'] + CORE_LABELS].rename(
        columns={l: f'{l}_csv' for l in CORE_LABELS}
    ),
    on='image_name', how='inner'
)

inconsistencies = 0
for l in CORE_LABELS:
    diff = (df_merge[l] != df_merge[f'{l}_csv']).sum()
    inconsistencies += diff
    if diff > 0:
        print(f'⚠ {l}: {diff} inconsistências entre CSV original e split')

if inconsistencies == 0:
    print('✓ Rótulos do CSV original e dos splits são consistentes.')

print(f'\nImagens só nos splits (não no CSV): {len(only_splits)}')
print(f'Imagens só no CSV (não nos splits): {len(only_csv)} — estas são as excluídas na limpeza')

## 3. Original image resolution analysis

In [ ]:
all_split_images = set()
for fold in range(5):
    for split in ['train', 'val', 'test']:
        df_s = pd.read_csv(SPLITS_DIR / f'fold_{fold}_{split}.csv')
        all_split_images |= set(df_s['image_name'])

print(f'Total de imagens únicas nos splits: {len(all_split_images)}')
print('Lendo dimensões... (pode levar alguns minutos)')

resolution_data = []
corrupted       = []

for fname in tqdm(sorted(all_split_images)):
    fpath = IMGS_DIR / fname
    try:
        with Image.open(fpath) as img:
            w, h = img.size
            mode = img.mode
            resolution_data.append({'image_name': fname, 'width': w, 'height': h,
                                     'mode': mode, 'aspect': round(w/h, 3)})
    except (UnidentifiedImageError, Exception) as e:
        corrupted.append({'image_name': fname, 'error': str(e)})

df_res = pd.DataFrame(resolution_data)
print(f'\nLidas com sucesso: {len(df_res)}')
print(f'Corrompidas      : {len(corrupted)}')
if corrupted:
    print('Imagens corrompidas:', [c['image_name'] for c in corrupted])

In [ ]:
print('=== RESOLUÇÃO ORIGINAL (antes de resize) ===')
print(f'\nLargura  — min:{df_res.width.min()}  max:{df_res.width.max()}  '
      f'mediana:{df_res.width.median():.0f}  média:{df_res.width.mean():.0f}')
print(f'Altura   — min:{df_res.height.min()}  max:{df_res.height.max()}  '
      f'mediana:{df_res.height.median():.0f}  média:{df_res.height.mean():.0f}')
print(f'\nModos de cor: {df_res['mode'].value_counts().to_dict()}')
print(f'\nAspect ratio — min:{df_res.aspect.min():.2f}  '
      f'max:{df_res.aspect.max():.2f}  mediana:{df_res.aspect.median():.2f}')

print(f'\nTop 10 resoluções mais frequentes:')
top_res = df_res.groupby(['width','height']).size().sort_values(ascending=False).head(10)
for (w,h), cnt in top_res.items():
    print(f'  {w}×{h}: {cnt} imagens ({100*cnt/len(df_res):.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.scatter(df_res['width'], df_res['height'], alpha=0.15, s=8, color='steelblue')
ax.axvline(224, color='red', linestyle='--', linewidth=1, label='224px (resize alvo)')
ax.axhline(224, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Largura (px)')
ax.set_ylabel('Altura (px)')
ax.set_title('Resolução original', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(df_res['width'], bins=30, color='steelblue', edgecolor='white')
ax.axvline(df_res['width'].median(), color='orange', linestyle='--',
           linewidth=1.5, label=f'Mediana={df_res["width"].median():.0f}')
ax.set_xlabel('Largura (px)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de larguras', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[2]
ax.hist(df_res['aspect'], bins=30, color='#4CAF50', edgecolor='white')
ax.axvline(1.0, color='red', linestyle='--', linewidth=1, label='1:1')
ax.set_xlabel('Aspect ratio (W/H)')
ax.set_ylabel('Frequência')
ax.set_title('Aspect ratio', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('Distribuição de resoluções das imagens originais', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'resolucao_distribuicao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 4. Visual quality analysis (brightness and contrast)

In [ ]:
import random
random.seed(42)
sample_imgs = random.sample(sorted(all_split_images), min(300, len(all_split_images)))

quality_data = []
print('Analisando qualidade visual de 300 imagens amostradas...')

for fname in tqdm(sample_imgs):
    fpath = IMGS_DIR / fname
    try:
        with Image.open(fpath) as img:
            arr  = np.array(img.convert('RGB'), dtype=np.float32)
            mean = float(arr.mean())
            std  = float(arr.std())

            flag_dark    = mean < 30
            flag_bright  = mean > 220
            flag_lowvar  = std < 10
            quality_data.append({
                'image_name': fname, 'mean': mean, 'std': std,
                'dark': flag_dark, 'bright': flag_bright, 'low_var': flag_lowvar
            })
    except Exception:
        pass

df_qual = pd.DataFrame(quality_data)
n_dark   = int(df_qual['dark'].sum())
n_bright = int(df_qual['bright'].sum())
n_lowvar = int(df_qual['low_var'].sum())

print(f'\nQualidade visual (amostra de {len(df_qual)} imagens):')
print(f'  Muito escuras  (mean < 30)  : {n_dark}  ({100*n_dark/len(df_qual):.1f}%)')
print(f'  Muito claras   (mean > 220) : {n_bright}  ({100*n_bright/len(df_qual):.1f}%)')
print(f'  Baixa variação (std < 10)   : {n_lowvar}  ({100*n_lowvar/len(df_qual):.1f}%)')
print(f'  Brilho médio geral          : {df_qual["mean"].mean():.1f} ± {df_qual["mean"].std():.1f}')
print(f'  Contraste médio geral       : {df_qual["std"].mean():.1f} ± {df_qual["std"].std():.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.hist(df_qual['mean'], bins=40, color='steelblue', edgecolor='white')
ax.axvline(30,  color='red',    linestyle='--', linewidth=1, alpha=0.7, label='Limite escura (30)')
ax.axvline(220, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Limite clara (220)')
ax.set_xlabel('Brilho médio (0–255)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de brilho médio', fontweight='bold')
ax.legend(fontsize=8)

ax = axes[1]
ax.hist(df_qual['std'], bins=40, color='#FF9800', edgecolor='white')
ax.axvline(10, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Limite baixa variação (10)')
ax.set_xlabel('Desvio padrão dos pixels')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de contraste (std pixels)', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('Qualidade visual — brilho e contraste', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'qualidade_brilho_contraste.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 5. Visual sample grid by class

In [ ]:
df_tr0 = pd.read_csv(SPLITS_DIR / 'fold_0_train.csv')

def show_class_grid(label: str, n_cols: int = 6, n_rows: int = 3,
                    img_size: int = 160, seed: int = 42):
    """
    Mostra grade de n_cols × n_rows amostras positivas de uma classe.
    """
    rng       = random.Random(seed)
    positives = df_tr0[df_tr0[label] == 1]['image_name'].tolist()
    n_show    = min(n_cols * n_rows, len(positives))
    selected  = rng.sample(positives, n_show)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.2, n_rows * 2.2))
    axes = axes.flatten()

    for ax, fname in zip(axes, selected):
        try:
            img = Image.open(IMGS_DIR / fname).convert('RGB')
            img = img.resize((img_size, img_size), RESAMPLING_METHOD)
            ax.imshow(img)
        except Exception:
            ax.set_facecolor('lightgray')
        ax.axis('off')
        ax.set_title(fname.replace('.jpg',''), fontsize=6, pad=1)

    for ax in axes[n_show:]:
        ax.axis('off')

    fig.suptitle(f'{label}  —  {len(positives)} imagens positivas no treino (fold 0)',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    fig.savefig(FIGS_DIR / f'amostras_{label.lower()}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'  → {label}: {len(positives)} positivos, {n_show} exibidos')

print('Gerando grades visuais por classe...\n')
for label in CORE_LABELS:
    show_class_grid(label)

In [ ]:
print('Artefatos:\n')
for label in ARTIFACT_LABELS:
    show_class_grid(label, n_cols=6, n_rows=2)

## 6. Shortcut risk visualization — LIGHT × EROSION

In [ ]:
def sample_group(mask, n=5, seed=42):
    rng = random.Random(seed)
    imgs = df_tr0[mask]['image_name'].tolist()
    return rng.sample(imgs, min(n, len(imgs)))

erosao_sem_luz = sample_group((df_tr0['EROSÃO']==1) & (df_tr0['LUZ']==0))
erosao_com_luz = sample_group((df_tr0['EROSÃO']==1) & (df_tr0['LUZ']==1))
luz_sem_erosao = sample_group((df_tr0['LUZ']==1)   & (df_tr0['EROSÃO']==0))

groups = [
    ('EROSÃO\n(sem LUZ)', erosao_sem_luz,  '#4CAF50'),
    ('EROSÃO + LUZ\n(risco shortcut)', erosao_com_luz, '#F44336'),
    ('LUZ\n(sem EROSÃO)', luz_sem_erosao, '#FF9800'),
]

n_per_group = 5
fig, axes = plt.subplots(n_per_group, 3, figsize=(9, n_per_group * 3))

for col, (title, fnames, color) in enumerate(groups):
    for row in range(n_per_group):
        ax = axes[row, col]
        if row < len(fnames):
            try:
                img = Image.open(IMGS_DIR / fnames[row]).convert('RGB')
                img = img.resize((200, 200), RESAMPLING_METHOD)
                ax.imshow(img)
                for spine in ax.spines.values():
                    spine.set_edgecolor(color)
                    spine.set_linewidth(3)
            except Exception:
                ax.set_facecolor('lightgray')
        else:
            ax.set_facecolor('white')
        ax.axis('off')
    axes[0, col].set_title(title, fontsize=10, fontweight='bold',
                            color=color, pad=8)

fig.suptitle('Risco de shortcut: EROSÃO × LUZ\n'
             'O modelo pode aprender que reflexo de luz → EROSÃO?',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'shortcut_erosao_luz.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura de shortcut salva.')

## 7. Visualization of multi-label images

In [ ]:
df_tr0['n_core'] = df_tr0[CORE_LABELS].sum(axis=1)
df_multi = df_tr0[df_tr0['n_core'] >= 2].copy()

print(f'Imagens com ≥2 rótulos clínicos no treino fold 0: {len(df_multi)}')
print('Distribuição de rótulos nessas imagens:')
print(df_multi[CORE_LABELS].sum().sort_values(ascending=False).to_string())

df_multi_sorted = df_multi.sort_values('n_core', ascending=False).head(12)

n_cols = 4
n_rows = 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
axes = axes.flatten()

for ax, (_, row) in zip(axes, df_multi_sorted.iterrows()):
    try:
        img = Image.open(IMGS_DIR / row['image_name']).convert('RGB')
        img = img.resize((200, 200), RESAMPLING_METHOD)
        ax.imshow(img)
    except Exception:
        ax.set_facecolor('lightgray')

    active = [l[:4] for l in CORE_LABELS if row[l] == 1]
    ax.set_title(' + '.join(active), fontsize=7.5, fontweight='bold', color='#1a237e')
    ax.axis('off')

for ax in axes[len(df_multi_sorted):]:
    ax.axis('off')

fig.suptitle('Imagens com ≥2 rótulos clínicos simultâneos (mais complexas)',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'amostras_multilabel.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura multilabel salva.')

## 8. Save consolidated report

In [ ]:
eda_report = {
    'metadata': {
        'notebook':  '01_eda_imagens',
        'csv_path':  str(CSV_PATH),
        'imgs_dir':  str(IMGS_DIR),
    },
    'csv_original': {
        'total_rows':       len(df),
        'valid_with_file':  len(df_valid),
        'orphan_csv_rows':  len(df) - len(df_valid),
        'label_stats':      label_stats,
    },
    'resolution': {
        'n_analyzed':     len(df_res),
        'corrupted':      len(corrupted),
        'corrupted_list': [c['image_name'] for c in corrupted],
        'width':  {'min': int(df_res.width.min()),  'max': int(df_res.width.max()),
                   'median': float(df_res.width.median()), 'mean': float(df_res.width.mean())},
        'height': {'min': int(df_res.height.min()), 'max': int(df_res.height.max()),
                   'median': float(df_res.height.median()), 'mean': float(df_res.height.mean())},
        'aspect': {'min': float(df_res.aspect.min()), 'max': float(df_res.aspect.max()),
                   'median': float(df_res.aspect.median())},
        'color_modes': df_res['mode'].value_counts().to_dict(),
        'top_resolutions': [
            {'width': int(w), 'height': int(h), 'count': int(cnt)}
            for (w, h), cnt in top_res.items()
        ],
    },
    'quality': {
        'n_sampled':    len(df_qual),
        'n_dark':       int(df_qual['dark'].sum()),
        'n_bright':     int(df_qual['bright'].sum()),
        'n_low_var':    int(df_qual['low_var'].sum()),
        'mean_brightness': float(df_qual['mean'].mean()),
        'mean_contrast':   float(df_qual['std'].mean()),
    },
    'multilabel': {
        'n_with_ge2_core': int(len(df_multi)),
        'pct_ge2_core':    round(100 * len(df_multi) / len(df_tr0), 2),
    },
    'shortcut_confirmed': {
        'erosao_com_luz': int(((df_tr0['EROSÃO']==1) & (df_tr0['LUZ']==1)).sum()),
        'erosao_total':   int((df_tr0['EROSÃO']==1).sum()),
        'pct':            round(100 * ((df_tr0['EROSÃO']==1) & (df_tr0['LUZ']==1)).sum() /
                                max(1, (df_tr0['EROSÃO']==1).sum()), 1),
    },
}

out_path = RESULTS_DIR / 'eda_report.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(eda_report, f, ensure_ascii=False, indent=2)

print(f'Relatório salvo em: {out_path}')
print()
print('=== RESUMO FINAL ===')
print(f'  Imagens válidas no CSV            : {len(df_valid)}')
print(f'  Imagens corrompidas               : {len(corrupted)}')
print(f'  Resolução mais comum              : {top_res.index[0][0]}×{top_res.index[0][1]} ({top_res.iloc[0]} imgs)')
print(f'  Imagens muito escuras/claras/pobres: {n_dark}/{n_bright}/{n_lowvar} (amostra de {len(df_qual)})')
print(f'  EROSÃO com LUZ (risco shortcut)   : {eda_report["shortcut_confirmed"]["pct"]}% das erosões')
print(f'  Imagens com ≥2 rótulos clínicos   : {len(df_multi)} ({eda_report["multilabel"]["pct_ge2_core"]}% do treino)')